# 第一步：化合物级标签聚合与 ChemVista 拆分可行性分析

本 Notebook 只做建模前的数据准备，不训练模型，也不根据任何模型性能选择外部测试集。

主要目标：

1. 将严格性质候选按 `IK2D + task` 聚合为化合物级标签；
2. 统计性质结构、公共谱库结构与 ChemVista 结构的交集；
3. 生成主任务样本量、二分类类别均衡和 ChemVista 拆分可行性报告；
4. 为下一步选择 `ChemVista-Novel`、`ChemVista-Transfer` 提供固定输入。

> 注意：所有样本量都按独立 IK2D 统计，不按性质记录行数或谱图数量统计。

## 1. 环境与项目路径

在 PyCharm 中将解释器设置为：`D:\\Appinstallation\\Anaconda\\envs\\env_ms\\python.exe`。

Notebook 可以从项目根目录或本文件所在的 `建模分析` 目录启动。

In [ ]:
from __future__ import annotations

import hashlib
import json
import pickle
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem.Scaffolds import MurckoScaffold

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '性质数据').exists() and (candidate / '质谱数据').exists():
            return candidate
    raise FileNotFoundError('未找到项目根目录：需要同时包含“性质数据”和“质谱数据”目录。')

ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'preparation_step1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('项目根目录:', ROOT)
print('输出目录:', OUTPUT_DIR)
print('Python:', sys.version.split()[0])
print('Python executable:', sys.executable)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('RDKit:', rdBase.rdkitVersion)
print('platform:', platform.platform())

## 2. 输入文件与任务定义

这里固定主任务，避免后续根据模型结果改变研究问题。

In [ ]:
PATHS = {
    'strict_targets': ROOT / '性质数据' / 'Cleaned_datasets' / '01_integrated_properties' / '07b_PBT_strict_MSMS_modeling_candidates.csv',
    'compound_master': ROOT / '性质数据' / 'Cleaned_datasets' / '01_integrated_properties' / '02_PBT_global_compound_master.csv',
    'public_index': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Index.pkl',
    'public_metadata': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Metadata.csv.gz',
    'public_summary': ROOT / '质谱数据' / 'EnvMolBench_Cleaned_Summary.json',
    'public_cleaning_notebook': ROOT / '质谱数据' / '质谱数据清洗.ipynb',
    'public_cleaning_module': ROOT / 'src' / 'pbt_msms' / 'public_spectrum_cleaning.py',
    'chemvista_spectrum_index': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_spectrum_index.pkl',
    'chemvista_compound_index': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_compound_index.pkl',
    'chemvista_peaks': ROOT / '验证实验谱图数据' / 'ChemVista_cleaned' / 'ChemVista_spectrum_peaks.pkl',
}

PRIMARY_TASKS = [
    'P_screen_not_ready_binary',
    'B_LOGBCF_REGRESSION',
    'T_EC50_MG_L_REGRESSION',
    'T_LC50_MG_L_REGRESSION',
    'T_NOEC_MG_L_REGRESSION',
]
SECONDARY_TASKS = [
    'P_READY_BOD_D28_OECD301C_SOURCE_301C',
    'T_EC10_MG_L_REGRESSION',
]
DESCRIPTIVE_ONLY_TASKS = [
    'P_formal_strict_binary',
    'vP_formal_strict_binary',
    'T_formal_strict_binary',
]
SELECTED_TASKS = PRIMARY_TASKS + SECONDARY_TASKS + DESCRIPTIVE_ONLY_TASKS

missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少输入文件：\n' + '\n'.join(missing)

pd.DataFrame({
    'key': PATHS.keys(),
    'path': [str(p.relative_to(ROOT)) for p in PATHS.values()],
    'size_mb': [round(p.stat().st_size / 1024**2, 3) for p in PATHS.values()],
})

## 3. 读取性质、公共谱库索引和 ChemVista

公共谱库读取轻量索引与V2逐谱图元数据，不读取多GB主谱图文件。Development主交集严格按 `identity_ppm_error <= 20` 定义；50/100 ppm只保留为敏感性计数。

In [ ]:
strict = pd.read_csv(PATHS['strict_targets'], low_memory=False)
master = pd.read_csv(PATHS['compound_master'], low_memory=False)

with PATHS['public_index'].open('rb') as handle:
    public_index: dict[str, list[int]] = pickle.load(handle)
with PATHS['public_summary'].open('r', encoding='utf-8') as handle:
    public_summary = json.load(handle)
assert public_summary.get('status') == 'PASS' and public_summary.get('schema_version') == 'ENVMOLBENCH_CLEANED_V2'
public_metadata = pd.read_csv(
    PATHS['public_metadata'],
    usecols=['inchikey_2d', 'identity_ppm_error', 'ppm_tier'],
    dtype={'inchikey_2d': 'string', 'ppm_tier': 'string'},
    low_memory=False,
)
public_metadata['inchikey_2d'] = public_metadata['inchikey_2d'].fillna('').str.strip()
public_metadata['identity_ppm_error'] = pd.to_numeric(public_metadata['identity_ppm_error'], errors='coerce')
public_counts_master = public_metadata.groupby('inchikey_2d', observed=True).size().rename('public_spectrum_count_master_100ppm')
public_counts_50 = public_metadata.loc[public_metadata['identity_ppm_error'].le(50)].groupby('inchikey_2d', observed=True).size().rename('public_spectrum_count_50ppm')
public_counts_primary = public_metadata.loc[public_metadata['identity_ppm_error'].le(20)].groupby('inchikey_2d', observed=True).size().rename('public_spectrum_count')
assert len(public_metadata) == sum(map(len, public_index.values())) == int(public_summary['counts']['SUCCESS'])
assert set(public_counts_master.index) == set(public_index)
assert public_metadata['identity_ppm_error'].notna().all() and public_metadata['identity_ppm_error'].le(100).all()

chem_spectra = pd.read_pickle(PATHS['chemvista_spectrum_index']).copy()
chem_compounds = pd.read_pickle(PATHS['chemvista_compound_index']).copy()

def normalize_ik2d(value) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip()
    return text[5:] if text.startswith('IK2D:') else text

strict['inchikey_2d'] = strict['global_identity_key'].map(normalize_ik2d)
strict['target_value_numeric'] = pd.to_numeric(strict['target_value'], errors='coerce')
master['inchikey_2d'] = master['inchikey_2d'].fillna('').astype(str).str.strip()
chem_spectra['inchikey_2d'] = chem_spectra['inchikey_2d'].fillna('').astype(str).str.strip()
chem_compounds['inchikey_2d'] = chem_compounds['inchikey_2d'].fillna('').astype(str).str.strip()

public_all_keys = set(public_index)
public_keys = set(public_counts_primary.index)
chem_keys = set(chem_compounds.loc[chem_compounds['inchikey_2d'].ne(''), 'inchikey_2d'])
property_keys = set(strict.loc[strict['inchikey_2d'].ne(''), 'inchikey_2d'])

print(f'严格性质候选：{len(strict):,} 行，{len(property_keys):,} 个IK2D')
print(f'公共谱库master(<=100 ppm)：{len(public_metadata):,} 张谱图，{len(public_all_keys):,} 个IK2D')
print(f'公共谱库主分析(<=20 ppm)：{int(public_counts_primary.sum()):,} 张谱图，{len(public_keys):,} 个IK2D')
print(f'ChemVista：{len(chem_spectra):,} 张谱图，{len(chem_keys):,} 个IK2D')
print(f'性质 ∩ 公共谱库：{len(property_keys & public_keys):,}')
print(f'性质 ∩ ChemVista：{len(property_keys & chem_keys):,}')
print(f'性质 ∩ 公共谱库 ∩ ChemVista：{len(property_keys & public_keys & chem_keys):,}')
print(f'性质 ∩ ChemVista，但不在公共谱库：{len((property_keys & chem_keys) - public_keys):,}')

## 4. 按 IK2D + 任务聚合性质标签

- 二分类：只有同一化合物的所有有效记录一致时才保留标签；否则标记冲突。
- 回归：主标签取中位数，同时保存最小值、最大值、四分位距、记录数和来源数。
- 不把多条研究记录当作多个独立化合物。

In [ ]:
HIGH_QUALITY_DATASETS = {'ECHA_B', 'ECHA_P', 'ECHA_T', 'NITE_BCF_B', 'NITE_BIODEG_P', 'OPENFOODTOX_T'}
MODEL_DEVELOPMENT_DATASETS = {'OPERA_PBT'}
LOWER_CONFIDENCE_TIERS = {'tier_2_unqualified_point_estimate'}
B_THRESHOLD_LOG10 = float(np.log10(2000.0))
VB_THRESHOLD_LOG10 = float(np.log10(5000.0))
AQUATIC_T_THRESHOLD_LOG10 = float(np.log10(0.01))

def aggregate_values(values: pd.Series, is_binary: bool) -> tuple[float, bool]:
    values = pd.to_numeric(values, errors='coerce').dropna()
    unique_values = sorted(values.unique().tolist())
    conflict = bool(is_binary and len(unique_values) > 1)
    if len(values) == 0 or conflict:
        return np.nan, conflict
    return (float(unique_values[0]) if is_binary else float(values.median())), conflict

def aggregate_one_group(group: pd.DataFrame) -> pd.Series:
    values = pd.to_numeric(group['target_value_numeric'], errors='coerce').dropna()
    target_type = str(group['target_type'].iloc[0])
    is_binary = target_type == 'binary'
    aggregate_value, conflict = aggregate_values(values, is_binary)
    dataset = group['dataset_id'].astype(str)
    evidence_tier = group['evidence_tier'].fillna('').astype(str)
    high_quality_mask = dataset.isin(HIGH_QUALITY_DATASETS) & ~evidence_tier.isin(LOWER_CONFIDENCE_TIERS)
    direct_experimental_mask = ~dataset.isin(MODEL_DEVELOPMENT_DATASETS)
    high_quality_values = pd.to_numeric(group.loc[high_quality_mask, 'target_value_numeric'], errors='coerce').dropna()
    experimental_values = pd.to_numeric(group.loc[direct_experimental_mask, 'target_value_numeric'], errors='coerce').dropna()
    high_quality_value, high_quality_conflict = aggregate_values(high_quality_values, is_binary)
    experimental_value, experimental_conflict = aggregate_values(experimental_values, is_binary)
    value_min = float(values.min()) if len(values) else np.nan
    value_max = float(values.max()) if len(values) else np.nan
    value_range = value_max - value_min if len(values) else np.nan
    iqr = float(values.quantile(0.75) - values.quantile(0.25)) if len(values) else np.nan
    target_name = str(group.name[1])
    threshold_conflict = False
    threshold_name = ''
    if target_name == 'B_LOGBCF_REGRESSION' and len(values):
        b_cross = value_min < B_THRESHOLD_LOG10 <= value_max
        vb_cross = value_min < VB_THRESHOLD_LOG10 <= value_max
        threshold_conflict = bool(b_cross or vb_cross)
        threshold_name = ';'.join(name for name, flag in [('B', b_cross), ('vB', vb_cross)] if flag)
    elif target_name in {'T_EC50_MG_L_REGRESSION', 'T_LC50_MG_L_REGRESSION', 'T_NOEC_MG_L_REGRESSION'} and len(values):
        threshold_conflict = bool(value_min <= AQUATIC_T_THRESHOLD_LOG10 < value_max)
        threshold_name = 'aquatic_T_0.01mg_L' if threshold_conflict else ''
    return pd.Series({
        'pbt_dimension': group['pbt_dimension'].iloc[0],
        'target_type': target_type,
        'aggregate_value': aggregate_value,
        'high_quality_value': high_quality_value,
        'direct_experimental_value': experimental_value,
        'conservative_min': float(values.min()) if len(values) else np.nan,
        'max_value': float(values.max()) if len(values) else np.nan,
        'value_range': value_range,
        'q25': float(values.quantile(0.25)) if len(values) else np.nan,
        'q75': float(values.quantile(0.75)) if len(values) else np.nan,
        'iqr': iqr,
        'high_iqr_flag': bool((not is_binary) and np.isfinite(iqr) and iqr > 1.0),
        'extreme_iqr_flag': bool((not is_binary) and np.isfinite(iqr) and iqr > 2.0),
        'threshold_conflict': threshold_conflict,
        'threshold_conflict_name': threshold_name,
        'n_records': int(len(group)),
        'n_high_quality_records': int(high_quality_mask.sum()),
        'n_direct_experimental_records': int(direct_experimental_mask.sum()),
        'n_sources': int(group['dataset_id'].nunique()),
        'source_datasets': ';'.join(sorted(group['dataset_id'].astype(str).unique())),
        'evidence_tiers': ';'.join(sorted(group['evidence_tier'].dropna().astype(str).unique())),
        'binary_conflict': conflict,
        'high_quality_binary_conflict': high_quality_conflict,
        'experimental_binary_conflict': experimental_conflict,
    })

selected = strict[strict['target_name'].isin(SELECTED_TASKS)].copy()
aggregated_long = (
    selected.groupby(['inchikey_2d', 'target_name'], sort=True, dropna=False)
    .apply(aggregate_one_group, include_groups=False)
    .reset_index()
)

assert not aggregated_long.duplicated(['inchikey_2d', 'target_name']).any()
aggregated_long.head()

In [ ]:
def pivot_metric(frame: pd.DataFrame, metric: str, prefix: str) -> pd.DataFrame:
    wide = frame.pivot(index='inchikey_2d', columns='target_name', values=metric)
    wide.columns = [f'{prefix}__{column}' for column in wide.columns]
    return wide

metric_specs = [
    ('aggregate_value', 'label'),
    ('high_quality_value', 'label_hq'),
    ('direct_experimental_value', 'label_experimental'),
    ('conservative_min', 'min'),
    ('max_value', 'max'),
    ('value_range', 'range'),
    ('q25', 'q25'),
    ('q75', 'q75'),
    ('iqr', 'iqr'),
    ('high_iqr_flag', 'high_iqr'),
    ('extreme_iqr_flag', 'extreme_iqr'),
    ('threshold_conflict', 'threshold_conflict'),
    ('n_records', 'n_records'),
    ('n_high_quality_records', 'n_hq_records'),
    ('n_direct_experimental_records', 'n_experimental_records'),
    ('n_sources', 'n_sources'),
    ('binary_conflict', 'binary_conflict'),
]
labels_wide = pd.concat(
    [pivot_metric(aggregated_long, metric, prefix) for metric, prefix in metric_specs],
    axis=1,
).reset_index()

print('聚合后具有至少一个选定任务标签的化合物：', len(labels_wide))
print('二分类冲突数：', int(aggregated_long['binary_conflict'].sum()))

## 5. 选择可追溯 SMILES 并计算 Murcko scaffold

全局主表中少数化合物可能有多个标准化 SMILES。优先选择重新生成的 IK2D 与主键一致的候选。非环化合物暂记为 `ACYCLIC`；下一步正式拆分时需要用 Morgan 结构聚类进一步拆分该组。

In [ ]:
def valid_smiles_candidates(value) -> list[str]:
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(';') if item.strip()]

def choose_smiles(value, expected_ik2d: str) -> tuple[str, str]:
    fallback = ''
    for candidate in valid_smiles_candidates(value):
        mol = Chem.MolFromSmiles(candidate)
        if mol is None:
            continue
        canonical = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)
        fallback = fallback or canonical
        generated = Chem.MolToInchiKey(mol).split('-')[0]
        if generated == expected_ik2d:
            return canonical, 'ik2d_verified'
    return fallback, ('fallback_valid_smiles' if fallback else 'missing_or_invalid')

def murcko_scaffold(smiles: str) -> str:
    if not smiles:
        return ''
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return ''
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    if scaffold.GetNumAtoms() == 0:
        return 'ACYCLIC'
    generic = MurckoScaffold.MakeScaffoldGeneric(scaffold)
    return Chem.MolToSmiles(generic, canonical=True, isomericSmiles=False)

master_structures = master.loc[
    master['inchikey_2d'].ne(''),
    ['inchikey_2d', 'canonical_smiles_2d_values', 'preferred_names', 'dataset_ids'],
].drop_duplicates('inchikey_2d').copy()

chosen = [choose_smiles(row.canonical_smiles_2d_values, row.inchikey_2d) for row in master_structures.itertuples()]
master_structures['canonical_smiles'] = [item[0] for item in chosen]
master_structures['smiles_resolution'] = [item[1] for item in chosen]
master_structures['murcko_scaffold_generic'] = master_structures['canonical_smiles'].map(murcko_scaffold)

master_structures['smiles_resolution'].value_counts(dropna=False)

## 6. 合并公共谱图和 ChemVista 可用性

这里生成的是拆分候选角色，还没有正式选择额外的 `ChemVista-Novel`。

In [ ]:
public_counts = pd.concat([public_counts_primary, public_counts_50, public_counts_master], axis=1).fillna(0).astype(int)
public_counts.index.name = 'inchikey_2d'

chem_summary = (
    chem_spectra[chem_spectra['inchikey_2d'].ne('')]
    .groupby('inchikey_2d')
    .agg(
        chemvista_spectrum_count=('spectrum_id', 'nunique'),
        chemvista_positive_count=('ion_mode', lambda x: int(x.astype(str).str.upper().eq('POSITIVE').sum())),
        chemvista_negative_count=('ion_mode', lambda x: int(x.astype(str).str.upper().eq('NEGATIVE').sum())),
        chemvista_precursor_mz_median=('precursor_mz', 'median'),
        chemvista_collision_energy_median=('collision_energy_value', 'median'),
        chemvista_peak_count_median=('cleaned_peak_n', 'median'),
        chemvista_instrument_types=('instrument_type', lambda x: ';'.join(sorted(set(x.dropna().astype(str))))),
    )
)

compound_table = labels_wide.merge(master_structures, on='inchikey_2d', how='left')
compound_table = compound_table.merge(public_counts, on='inchikey_2d', how='left')
compound_table = compound_table.merge(chem_summary, on='inchikey_2d', how='left')
for column in ['public_spectrum_count', 'public_spectrum_count_50ppm', 'public_spectrum_count_master_100ppm']:
    compound_table[column] = compound_table[column].fillna(0).astype(int)
compound_table['chemvista_spectrum_count'] = compound_table['chemvista_spectrum_count'].fillna(0).astype(int)
compound_table['has_public_spectrum'] = compound_table['public_spectrum_count'].gt(0)
compound_table['has_chemvista_spectrum'] = compound_table['chemvista_spectrum_count'].gt(0)
compound_table['natural_chemvista_novel'] = compound_table['has_chemvista_spectrum'] & ~compound_table['has_public_spectrum']

conditions = [
    compound_table['natural_chemvista_novel'],
    compound_table['has_chemvista_spectrum'] & compound_table['has_public_spectrum'],
    compound_table['has_public_spectrum'],
]
roles = [
    'chemvista_natural_novel',
    'chemvista_public_overlap_candidate',
    'development_public_only_candidate',
]
compound_table['split_candidate_role'] = np.select(conditions, roles, default='property_without_spectrum')

assert compound_table['inchikey_2d'].is_unique
compound_table['split_candidate_role'].value_counts()

## 7. 样本量与类别均衡报告

同时展示“全部ChemVista留出”的极端方案，帮助判断开发集会缩小到什么程度。

In [ ]:
subset_masks = {
    'property_all': pd.Series(True, index=compound_table.index),
    'public_overlap': compound_table['has_public_spectrum'],
    'chemvista_all': compound_table['has_chemvista_spectrum'],
    'development_full_chemvista_holdout': compound_table['has_public_spectrum'] & ~compound_table['has_chemvista_spectrum'],
    'chemvista_natural_novel': compound_table['natural_chemvista_novel'],
}

task_rows = []
balance_rows = []
target_types = aggregated_long.drop_duplicates('target_name').set_index('target_name')['target_type'].to_dict()
dimensions = aggregated_long.drop_duplicates('target_name').set_index('target_name')['pbt_dimension'].to_dict()

for task in SELECTED_TASKS:
    label_col = f'label__{task}'
    if label_col not in compound_table.columns:
        continue
    for subset_name, mask in subset_masks.items():
        values = pd.to_numeric(compound_table.loc[mask, label_col], errors='coerce').dropna()
        task_rows.append({
            'task_name': task,
            'pbt_dimension': dimensions.get(task, ''),
            'target_type': target_types.get(task, ''),
            'subset': subset_name,
            'n_compounds': int(len(values)),
        })
        if target_types.get(task) == 'binary':
            positive = int(values.eq(1).sum())
            negative = int(values.eq(0).sum())
            balance_rows.append({
                'task_name': task,
                'subset': subset_name,
                'n_compounds': int(len(values)),
                'positive': positive,
                'negative': negative,
                'positive_rate': positive / len(values) if len(values) else np.nan,
                'majority_accuracy': max(positive, negative) / len(values) if len(values) else np.nan,
            })

task_sample_counts = pd.DataFrame(task_rows)
class_balance = pd.DataFrame(balance_rows)

display(task_sample_counts.pivot(index='task_name', columns='subset', values='n_compounds').fillna(0).astype(int))
display(class_balance[class_balance['subset'].isin(['public_overlap', 'chemvista_all', 'development_full_chemvista_holdout'])])

label_quality_summary = (
    selected.groupby(['target_name', 'dataset_id', 'evidence_tier'], dropna=False)
    .agg(n_records=('model_target_id', 'size'), n_compounds=('inchikey_2d', 'nunique'))
    .reset_index()
)
label_uncertainty_report = aggregated_long.loc[
    aggregated_long['high_iqr_flag'] | aggregated_long['extreme_iqr_flag'] | aggregated_long['threshold_conflict'],
].copy()
toxicity_extreme_record_review = selected.loc[
    selected['target_name'].isin({'T_EC50_MG_L_REGRESSION', 'T_LC50_MG_L_REGRESSION', 'T_NOEC_MG_L_REGRESSION'})
    & ((selected['target_value_numeric'] < -6.0) | (selected['target_value_numeric'] > 6.0)),
    ['model_target_id', 'inchikey_2d', 'target_name', 'target_value_numeric', 'dataset_id', 'evidence_tier', 'source_record_id', 'source_file'],
].copy()
display(label_quality_summary)
display(label_uncertainty_report[['target_name', 'high_iqr_flag', 'extreme_iqr_flag', 'threshold_conflict']].sum(numeric_only=True))
print('需人工复核的极端水生毒性原始记录：', len(toxicity_extreme_record_review))

## 8. 质量控制与输出

本步骤不生成最终拆分，只输出下一步所需的固定候选表。CSV使用UTF-8 BOM，便于Excel直接打开中文。

In [ ]:
def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

qc_checks = []
def add_check(check_id: str, description: str, passed: bool, details: str = ''):
    qc_checks.append({
        'check_id': check_id,
        'description': description,
        'status': 'PASS' if passed else 'FAIL',
        'details': details,
    })

add_check('STEP1_001', '化合物总表IK2D唯一', compound_table['inchikey_2d'].is_unique, str(len(compound_table)))
add_check('STEP1_002', '所有IK2D均为14位', compound_table['inchikey_2d'].str.fullmatch(r'[A-Z]{14}').all())
add_check('STEP1_003', '聚合任务键唯一', not aggregated_long.duplicated(['inchikey_2d', 'target_name']).any())
add_check('STEP1_004', '二分类聚合值仅为0/1/缺失', aggregated_long.loc[aggregated_long['target_type'].eq('binary'), 'aggregate_value'].dropna().isin([0, 1]).all())
add_check('STEP1_005', '公共谱库交集非空', int(compound_table['has_public_spectrum'].sum()) > 0, str(int(compound_table['has_public_spectrum'].sum())))
add_check('STEP1_006', 'ChemVista性质交集非空', int(compound_table['has_chemvista_spectrum'].sum()) > 0, str(int(compound_table['has_chemvista_spectrum'].sum())))
add_check('STEP1_007', '正式T不列入主任务', 'T_formal_strict_binary' not in PRIMARY_TASKS)
add_check('STEP1_008', '正式P/vP不列入主任务', not set(['P_formal_strict_binary', 'vP_formal_strict_binary']) & set(PRIMARY_TASKS))
regression_aggregates = aggregated_long[aggregated_long['target_type'].eq('regression') & aggregated_long['aggregate_value'].notna()]
add_check('STEP1_009', '回归中位标签均位于原始最小值和最大值之间', regression_aggregates['aggregate_value'].between(regression_aggregates['conservative_min'], regression_aggregates['max_value']).all())
add_check('STEP1_010', '高IQR标记与IQR阈值一致', aggregated_long['high_iqr_flag'].eq(aggregated_long['target_type'].eq('regression') & aggregated_long['iqr'].gt(1.0)).all())
add_check('STEP1_011', '极高IQR标记与IQR阈值一致', aggregated_long['extreme_iqr_flag'].eq(aggregated_long['target_type'].eq('regression') & aggregated_long['iqr'].gt(2.0)).all())
add_check('STEP1_012', '标签聚合未静默丢失来源记录', int(aggregated_long['n_records'].sum()) == len(selected), f"{int(aggregated_long['n_records'].sum())}/{len(selected)}")
add_check('STEP1_013', '低置信ECOTOX记录未进入高质量敏感性层', not selected.loc[selected['dataset_id'].eq('ECOTOX_T') & selected['evidence_tier'].isin(LOWER_CONFIDENCE_TIERS), 'dataset_id'].isin(HIGH_QUALITY_DATASETS).any())
add_check('STEP1_014', '公共谱图V2清洗摘要通过', public_summary.get('status') == 'PASS' and public_summary.get('schema_version') == 'ENVMOLBENCH_CLEANED_V2')
add_check('STEP1_015', '公共V2元数据、索引与摘要记录数守恒', len(public_metadata) == sum(map(len, public_index.values())) == int(public_summary['counts']['SUCCESS']))
add_check('STEP1_016', 'Development公共交集严格使用20ppm资格', compound_table['has_public_spectrum'].eq(compound_table['public_spectrum_count'].gt(0)).all() and public_keys == set(public_metadata.loc[public_metadata['identity_ppm_error'].le(20), 'inchikey_2d']))

qc = pd.DataFrame(qc_checks)
assert qc['status'].eq('PASS').all(), qc[qc['status'].eq('FAIL')].to_dict('records')

chemvista_feasibility = compound_table[compound_table['has_chemvista_spectrum']].copy()
chemvista_hashes = pd.DataFrame([
    {
        'file_key': key,
        'relative_path': str(PATHS[key].relative_to(ROOT)),
        'size_bytes': PATHS[key].stat().st_size,
        'sha256': sha256_file(PATHS[key]),
    }
    for key in ['chemvista_spectrum_index', 'chemvista_compound_index', 'chemvista_peaks']
])
public_input_hashes = pd.DataFrame([
    {
        'file_key': key,
        'relative_path': str(PATHS[key].relative_to(ROOT)),
        'size_bytes': PATHS[key].stat().st_size,
        'sha256': sha256_file(PATHS[key]),
    }
    for key in ['public_index', 'public_metadata', 'public_summary', 'public_cleaning_notebook', 'public_cleaning_module']
])

compound_table.to_csv(OUTPUT_DIR / 'compound_master_modeling.csv', index=False, encoding='utf-8-sig')
aggregated_long.to_csv(OUTPUT_DIR / 'compound_labels_long.csv', index=False, encoding='utf-8-sig')
chemvista_feasibility.to_csv(OUTPUT_DIR / 'chemvista_split_feasibility.csv', index=False, encoding='utf-8-sig')
task_sample_counts.to_csv(OUTPUT_DIR / 'task_sample_counts.csv', index=False, encoding='utf-8-sig')
class_balance.to_csv(OUTPUT_DIR / 'class_balance_report.csv', index=False, encoding='utf-8-sig')
label_quality_summary.to_csv(OUTPUT_DIR / 'label_quality_summary.csv', index=False, encoding='utf-8-sig')
label_uncertainty_report.to_csv(OUTPUT_DIR / 'label_uncertainty_report.csv', index=False, encoding='utf-8-sig')
toxicity_extreme_record_review.to_csv(OUTPUT_DIR / 'toxicity_extreme_record_review.csv', index=False, encoding='utf-8-sig')
qc.to_csv(OUTPUT_DIR / 'step1_qc_checks.csv', index=False, encoding='utf-8-sig')
chemvista_hashes.to_csv(OUTPUT_DIR / 'chemvista_input_hashes.csv', index=False, encoding='utf-8-sig')
public_input_hashes.to_csv(OUTPUT_DIR / 'public_input_hashes.csv', index=False, encoding='utf-8-sig')

audit = {
    'analysis_version': 'PBT_MSMS_PREPARATION_STEP1_V3',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(ROOT),
    'python_executable': sys.executable,
    'python_version': sys.version,
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'rdkit_version': rdBase.rdkitVersion,
    'strict_property_rows': int(len(strict)),
    'strict_property_unique_ik2d': int(len(property_keys)),
    'public_spectrum_count': int(public_counts_primary.sum()),
    'public_unique_ik2d': int(len(public_keys)),
    'public_master_100ppm_spectrum_count': int(len(public_metadata)),
    'public_master_100ppm_unique_ik2d': int(len(public_all_keys)),
    'public_primary_20ppm_spectrum_count': int(public_counts_primary.sum()),
    'public_primary_20ppm_unique_ik2d': int(len(public_keys)),
    'chemvista_spectrum_count': int(len(chem_spectra)),
    'chemvista_unique_ik2d': int(len(chem_keys)),
    'property_public_overlap': int(len(property_keys & public_keys)),
    'property_chemvista_overlap': int(len(property_keys & chem_keys)),
    'property_public_chemvista_overlap': int(len(property_keys & public_keys & chem_keys)),
    'property_chemvista_natural_novel': int(len((property_keys & chem_keys) - public_keys)),
    'selected_task_compound_count': int(len(compound_table)),
    'high_iqr_compound_tasks': int(aggregated_long['high_iqr_flag'].sum()),
    'extreme_iqr_compound_tasks': int(aggregated_long['extreme_iqr_flag'].sum()),
    'threshold_conflict_compound_tasks': int(aggregated_long['threshold_conflict'].sum()),
    'toxicity_extreme_records_for_manual_review': int(len(toxicity_extreme_record_review)),
    'label_sensitivity_layers': ['primary_median', 'conservative_min', 'direct_experimental_median', 'high_quality_median'],
    'qc_status': 'PASS',
}
with (OUTPUT_DIR / 'step1_audit.json').open('w', encoding='utf-8') as handle:
    json.dump(audit, handle, ensure_ascii=False, indent=2)
    handle.write('\n')

print('全部质量检查通过。')
print('已输出：')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name, f'({path.stat().st_size / 1024:.1f} KB)')
display(qc)
display(pd.Series(audit, name='value').to_frame())

## 9. 如何解读本步骤结果

运行结束后，优先查看：

1. `task_sample_counts.csv`：不同拆分假设下，各任务剩余多少独立化合物；
2. `class_balance_report.csv`：P、B/vB、正式P/T的类别分布；
3. `chemvista_split_feasibility.csv`：下一步选择 Novel/Transfer 的唯一输入；
4. `step1_qc_checks.csv`：必须全部为 PASS。

下一步才是根据结构簇和任务覆盖，固定选择 `ChemVista-Novel`，并生成不可更改的拆分清单。